# Step 5: Fine-tune ReactionT5 on ORD (Model 1, Colab GPU)

Runs `scripts/train_reactant_model_ord.py`, which fine-tunes the ORD-pretrained
checkpoint `sagawa/ReactionT5v2-retrosynthesis` (before any USPTO-specific
fine-tuning) on the freshly built, leak-checked `data/v2_ord_train/reactants_train.jsonl`.

**Colab session budget: ~3h/day.** Checkpoints are written to Google Drive, and this
notebook can simply be re-run on a later day -- it auto-resumes from the last
checkpoint. Do not clear the Drive folder between sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

`data/v2_ord_train/` and `data/v2_uspto_train/` are gitignored (large, derived) --
regenerate them deterministically here (fixed seed, excludes the committed
`data/v2_ord_eval_targets.json`/`data/v2_uspto_eval_targets.json` by construction,
so this can never leak into the eval sets). Only needs to run once per Colab
session; skipped automatically if the files already exist (e.g. this is a
same-day resume).

In [ ]:
import os

if not os.path.exists("data/v2_ord_train/reactants_train.jsonl"):
    !python scripts/build_train_data_ord.py --pool-count 60000 --seed 42
if not os.path.exists("data/v2_uspto_train/reactants_train.jsonl"):
    !python scripts/build_train_data_uspto.py

In [ ]:
output_dir = "/content/drive/MyDrive/retro-planner-checkpoints/model1_reactant"  # @param {type:"string"}
time_budget_minutes = 165  # @param {type:"number"}
mix_in_uspto = True  # @param {type:"boolean"}

extra_flag = ["--extra-train-file", "data/v2_uspto_train/reactants_train.jsonl"] if mix_in_uspto else []

In [ ]:
!python scripts/train_reactant_model_ord.py \
    --output-dir "{output_dir}" \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(extra_flag)}

Re-run the cell above (same `output_dir`) on the next day's Colab session to
continue training -- it auto-detects and resumes from the latest checkpoint.
Once `trainer.train()` finishes (not just time-budget-stopped), the final
model is also saved to `{output_dir}/final`.